In [ ]:
from model_fitting import fit_compare_to_threshold_model
from nwb_utils import NWBUtils

nwb_file_name='/root/capsule/scratch/general_behavior/698694/698694_2024-04-17_12-21-55.nwb'
nwb_data=NWBUtils.read_behavior_nwb(nwb_full_path=nwb_file_name)


In [ ]:
fitted_data=fit_compare_to_threshold_model(nwb_data,
                                        reset_on_switch=True,
                                        include_bias=True,
                                        save_results=True,
                                        save_folder='/root/capsule/scratch/CTT_reset_bias'
                                        )

In [ ]:
from __future__ import annotations

import os
import gc
import traceback
from pathlib import Path
import concurrent.futures as cf
from typing import Dict, Any, List

from nwb_utils import NWBUtils
from model_fitting import fit_compare_to_threshold_model


NWB_ROOT = Path("/root/capsule/scratch/general_behavior")
OUT_ROOT = Path("/root/capsule/scratch/CTT_grid_json_only_by_animal")

RESET_OPTIONS = [True, False]
BIAS_OPTIONS = [True, False]


def combo_outdir(out_root: Path, animal_id: str, reset_on_switch: bool, include_bias: bool) -> Path:
    return out_root / str(animal_id) / f"reset_on_switch_{reset_on_switch}" / f"include_bias_{include_bias}"


def find_all_nwbs_two_level(root: Path) -> List[Path]:
    nwb_files: List[Path] = []
    for animal_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        nwb_files.extend(sorted(animal_dir.glob("*.nwb")))
    return nwb_files


def extract_animal_id(nwb_path: Path) -> str:
    return nwb_path.parent.name


def run_one_file(nwb_path: Path) -> Dict[str, Any]:
    animal_id = extract_animal_id(nwb_path)

    status: Dict[str, Any] = {
        "file": str(nwb_path),
        "animal_id": animal_id,
        "ok": True,
        "errors": [],
        "results": [],  # keep only small summaries
    }

    nwb_data = None
    try:
        nwb_data = NWBUtils.read_behavior_nwb(nwb_full_path=str(nwb_path))
    except Exception as e:
        status["ok"] = False
        status["errors"].append(
            {"stage": "read_nwb", "error": repr(e), "traceback": traceback.format_exc()}
        )
        return status

    try:
        for reset_on_switch in RESET_OPTIONS:
            for include_bias in BIAS_OPTIONS:
                out_dir = combo_outdir(OUT_ROOT, animal_id, reset_on_switch, include_bias)
                out_dir.mkdir(parents=True, exist_ok=True)

                model_name = f"CTT_reset{int(reset_on_switch)}_bias{int(include_bias)}"

                try:
                    fit_out = fit_compare_to_threshold_model(
                        nwb_data,
                        model_name=model_name,
                        reset_on_switch=reset_on_switch,
                        include_bias=include_bias,
                        save_results=True,
                        save_folder=out_dir,
                    )

                    if fit_out is None:
                        status["ok"] = False
                        status["errors"].append(
                            {
                                "stage": "fit",
                                "reset_on_switch": reset_on_switch,
                                "include_bias": include_bias,
                                "error": "fit returned None",
                            }
                        )
                    else:
                        status["results"].append(
                            {
                                "reset_on_switch": reset_on_switch,
                                "include_bias": include_bias,
                                "session_id": fit_out.get("session_id", "unknown"),
                                "auto_train_stage": fit_out.get("auto_train_stage", "unknown"),
                                "aic": fit_out.get("aic", None),
                                "bic": fit_out.get("bic", None),
                                "neg_log_likelihood": fit_out.get("neg_log_likelihood", None),
                            }
                        )

                    # Drop fit output immediately
                    del fit_out

                except Exception as e:
                    status["ok"] = False
                    status["errors"].append(
                        {
                            "stage": "fit_exception",
                            "reset_on_switch": reset_on_switch,
                            "include_bias": include_bias,
                            "error": repr(e),
                            "traceback": traceback.format_exc(),
                        }
                    )

        return status

    finally:
        # Ensure we release references and run GC.
        # This helps, but true release depends on NWB/HDF5 closing behavior.
        if nwb_data is not None:
            del nwb_data
        gc.collect()


OUT_ROOT.mkdir(parents=True, exist_ok=True)

nwb_files = find_all_nwbs_two_level(NWB_ROOT)
print(f"Found {len(nwb_files)} NWB files under: {NWB_ROOT}")
print(f"Saving JSON results under: {OUT_ROOT}")

max_workers = min(6, len(nwb_files), os.cpu_count() or 1)
print(f"Running with {max_workers} threads")

n_ok = 0
n_fail = 0

with cf.ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(run_one_file, p): str(p) for p in nwb_files}

    for fut in cf.as_completed(futures):
        fpath = futures[fut]
        try:
            res = fut.result()
        except Exception as e:
            n_fail += 1
            print(f"[CRASH] {fpath} -> {repr(e)}")
            continue

        if res.get("ok", False):
            n_ok += 1
            print(f"[OK]   {res['animal_id']}  {fpath}")
        else:
            n_fail += 1
            print(f"[FAIL] {res['animal_id']}  {fpath}")
            for err in res.get("errors", [])[:2]:
                print("   ", err.get("stage", "unknown"), err.get("error", ""))
            if len(res.get("errors", [])) > 2:
                print("    ...")

print(f"Done. OK={n_ok}, FAIL={n_fail}")


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

import pandas as pd


def load_ctt_results_jsons_to_dataframe(
    results_root: Union[str, Path],
    *,
    recursive: bool = True,
    keep_latents: bool = False,
) -> pd.DataFrame:
    """
    Load all compare-to-threshold JSON result files under a results folder and
    concatenate them into a single pandas DataFrame.

    This is designed for the folder structure produced by your runner:
        CTT_grid_json_only_by_animal/
            <animal_id>/
                reset_on_switch_{True|False}/
                    include_bias_{True|False}/
                        <session_id>__<model_name>_fit_results.json

    Parameters
    ----------
    results_root
        Path to the root folder (e.g. /root/capsule/scratch/CTT_grid_json_only_by_animal).
    recursive
        If True, searches recursively for all *.json files under results_root.
        If False, only searches one level below results_root (not recommended here).
    keep_latents
        If True, keep the fitted_latent_variables field (which can be large) as
        a column containing dicts.
        If False, drop fitted_latent_variables to keep the DataFrame light-weight.

    Returns
    -------
    pd.DataFrame
        One row per JSON file. Nested dicts are flattened into columns using
        dotted keys, e.g.:
            fitted_params.alpha
            metadata.reset_on_switch

        If keep_latents=True, fitted_latent_variables is kept as a dict column
        (not flattened by default, since it can be huge).
    """

    root = Path(results_root)

    if not root.exists():
        raise FileNotFoundError(f"results_root does not exist: {root}")

    if recursive:
        json_files = sorted(root.rglob("*.json"))
    else:
        json_files = sorted(root.glob("*.json"))

    records: List[Dict[str, Any]] = []

    for jf in json_files:
        try:
            with open(jf, "r") as f:
                data = json.load(f)

            # Attach path-derived metadata that is useful for grouping
            # Expected layout: root / animal_id / reset_on_switch_X / include_bias_Y / file.json
            rel_parts = jf.relative_to(root).parts
            animal_id = rel_parts[0] if len(rel_parts) >= 1 else None

            reset_on_switch = None
            include_bias = None
            if len(rel_parts) >= 3:
                # e.g. "reset_on_switch_True"
                if rel_parts[1].startswith("reset_on_switch_"):
                    reset_on_switch = rel_parts[1].split("reset_on_switch_", 1)[1]
                    if reset_on_switch in ("True", "False"):
                        reset_on_switch = (reset_on_switch == "True")

                # e.g. "include_bias_False"
                if rel_parts[2].startswith("include_bias_"):
                    include_bias = rel_parts[2].split("include_bias_", 1)[1]
                    if include_bias in ("True", "False"):
                        include_bias = (include_bias == "True")

            data["_json_path"] = str(jf)
            data["_animal_id_from_path"] = animal_id
            data["_reset_on_switch_from_path"] = reset_on_switch
            data["_include_bias_from_path"] = include_bias

            # Optionally drop latents to avoid huge DF
            if not keep_latents and "fitted_latent_variables" in data:
                data.pop("fitted_latent_variables", None)

            records.append(data)

        except Exception as e:
            # Keep a minimal record for debugging failed loads
            records.append(
                {
                    "_json_path": str(jf),
                    "_load_error": repr(e),
                }
            )

    # Flatten nested dicts (except fitted_latent_variables if kept)
    df = pd.json_normalize(records, sep=".")

    # Optional: make common columns more convenient (if they exist)
    # If the JSON already contains these, the path-derived ones can be used as backup.
    if "animal_id" not in df.columns and "_animal_id_from_path" in df.columns:
        df["animal_id"] = df["_animal_id_from_path"]

    # If you want a single authoritative reset/include columns, prefer JSON metadata if present
    if "metadata.reset_on_switch" in df.columns and "_reset_on_switch_from_path" in df.columns:
        df["reset_on_switch"] = df["metadata.reset_on_switch"].where(
            df["metadata.reset_on_switch"].notna(), df["_reset_on_switch_from_path"]
        )
    elif "_reset_on_switch_from_path" in df.columns:
        df["reset_on_switch"] = df["_reset_on_switch_from_path"]

    if "metadata.include_bias" in df.columns and "_include_bias_from_path" in df.columns:
        df["include_bias"] = df["metadata.include_bias"].where(
            df["metadata.include_bias"].notna(), df["_include_bias_from_path"]
        )
    elif "_include_bias_from_path" in df.columns:
        df["include_bias"] = df["_include_bias_from_path"]

    return df


# Example usage:
# df = load_ctt_results_jsons_to_dataframe(
#     "/root/capsule/scratch/CTT_grid_json_only_by_animal",
#     keep_latents=False,
# )
# df.head()
